<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-09-toy-diffusion-understood-end-to-end.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9 (graded) — Toy diffusion, understood end to end
**Course 1: Hands-On Deep Learning with Python — Chapter 9: Diffusion foundations**

**Problem brief (Sam Okafor, Fernwood Media):** "A quick 'mood board' generator — we want
to understand what's under the hood before we adopt a vendor."

**What you'll submit:** the forward noising process and training loss implemented from
scratch, a trained small U-Net DDPM, a sample grid across training checkpoints, and a
one-page explainer for the art director.

## 1. Data: Fashion-MNIST at 28x28 (no offline fallback needed — torchvision ships it)

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 28

transform = T.Compose([T.ToTensor(), T.Normalize([0.5], [0.5])])  # scale to [-1, 1]
fmnist = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(fmnist, batch_size=128, shuffle=True, drop_last=True)

## 2. The forward (noising) process
Fill in the `TODO`: given a clean image `x0`, a timestep `t`, and noise `eps ~ N(0, I)`,
compute `x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps`.

In [ ]:
T_STEPS = 300
betas = torch.linspace(1e-4, 0.02, T_STEPS).to(device)           # noise schedule
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)                          # \bar\alpha_t

def forward_diffusion(x0, t, eps=None):
    """x0: (B, 1, H, W); t: (B,) int64 timesteps; returns (x_t, eps)."""
    if eps is None:
        eps = torch.randn_like(x0)
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    # TODO: x_t = sqrt(ab) * x0 + sqrt(1 - ab) * eps
    x_t = None  # replace with your implementation
    return x_t, eps

# sanity check once implemented: t=0 should look like x0, t=T-1 should look like pure noise
sample_x0, _ = next(iter(loader))
sample_x0 = sample_x0[:4].to(device)
for t_val in [0, T_STEPS // 2, T_STEPS - 1]:
    t = torch.full((4,), t_val, device=device, dtype=torch.long)
    x_t, _ = forward_diffusion(sample_x0, t)
    if x_t is not None:
        print(f't={t_val}: x_t std = {x_t.std().item():.3f} (should grow toward ~1 as t -> T)')

## 3. A small U-Net with a timestep embedding

In [ ]:
class TimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (9.2 / half))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        return self.mlp(emb)


class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, t_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, c_out)
        self.norm1 = nn.GroupNorm(8, c_out)
        self.norm2 = nn.GroupNorm(8, c_out)

    def forward(self, x, t_emb):
        h = torch.relu(self.norm1(self.conv1(x)))
        h = h + self.t_proj(t_emb).unsqueeze(-1).unsqueeze(-1)  # inject "how noisy is this"
        h = torch.relu(self.norm2(self.conv2(h)))
        return h


class TinyUNet(nn.Module):
    def __init__(self, t_dim=64):
        super().__init__()
        self.t_embed = TimestepEmbedding(t_dim)
        self.down1 = ConvBlock(1, 32, t_dim)
        self.pool = nn.MaxPool2d(2)
        self.down2 = ConvBlock(32, 64, t_dim)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.up1 = ConvBlock(64 + 32, 32, t_dim)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x, t):
        t_emb = self.t_embed(t)
        d1 = self.down1(x, t_emb)              # 28x28
        d2 = self.down2(self.pool(d1), t_emb)    # 14x14
        u1 = self.up(d2)                          # 28x28
        u1 = self.up1(torch.cat([u1, d1], dim=1), t_emb)
        return self.out(u1)

model = TinyUNet().to(device)

## 4. Training loss: predict the noise
Fill in the `TODO`: MSE between the network's predicted noise and the actual `eps` used in
the forward process.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
N_EPOCHS = 10
checkpoints = {}

for epoch in range(N_EPOCHS):
    for x0, _ in loader:
        x0 = x0.to(device)
        t = torch.randint(0, T_STEPS, (x0.size(0),), device=device)
        x_t, eps = forward_diffusion(x0, t)
        if x_t is None:
            raise NotImplementedError('Fill in the TODO in forward_diffusion (Section 2) above.')
        eps_pred = model(x_t, t)

        optimizer.zero_grad()
        # TODO: loss = MSE(eps_pred, eps)
        loss = None  # replace with your implementation
        if loss is None:
            raise NotImplementedError('Fill in the TODO: loss = MSE(eps_pred, eps) above.')
        loss.backward()
        optimizer.step()
    if loss is not None:
        print(f'epoch {epoch}: loss = {loss.item():.4f}')
    if epoch in (0, N_EPOCHS // 2, N_EPOCHS - 1):
        checkpoints[epoch] = {k: v.clone() for k, v in model.state_dict().items()}

## 5. DDPM sampling: start from pure noise, denoise step by step

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples=8):
    x = torch.randn(n_samples, 1, IMG_SIZE, IMG_SIZE, device=device)
    for t_val in reversed(range(T_STEPS)):
        t = torch.full((n_samples,), t_val, device=device, dtype=torch.long)
        eps_pred = model(x, t)
        alpha_t, alpha_bar_t, beta_t = alphas[t_val], alpha_bars[t_val], betas[t_val]
        mean = (1 / alpha_t.sqrt()) * (x - (beta_t / (1 - alpha_bar_t).sqrt()) * eps_pred)
        if t_val > 0:
            x = mean + beta_t.sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(checkpoints), figsize=(4 * len(checkpoints), 4))
for ax, (ep, state) in zip(axes if len(checkpoints) > 1 else [axes], checkpoints.items()):
    model.load_state_dict(state)
    samples = ddpm_sample(model, n_samples=8).cpu()
    grid = torchvision.utils.make_grid(samples, nrow=4, normalize=True)
    ax.imshow(grid.permute(1, 2, 0)); ax.set_title(f'after epoch {ep}'); ax.axis('off')
plt.suptitle('Sample gallery across training checkpoints')
plt.show()

model.load_state_dict(checkpoints[max(checkpoints)])  # end on the fully-trained model

## 6. One-page explainer for the art director (fill in)
In plain, non-technical language: what does this system actually do, step by step? What can
it not yet do that a real product (Stable Diffusion / Midjourney) can — text conditioning is
the big one, covered in Course 2's diffusion chapter, not built here. What's the honest
compute-cost reality of scaling this up?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 9: Diffusion foundations*